# Training a new model in the same architecture but with less data and testing and comparing it across original model.

## Goal
Test how the pre-trained model performs against the different metrics

## Metrics
(Similiar to the classes in the Study)
- Class 6: Hand-held, Back-pocket, Front-pocket, Coat-pocket, Lower-Back, Shoulder-Bag
- Class 5: Hand-held, Pocket (Back and Front combined), Coat-pocket, Lower-Back, Shoulder-Bag
- Class 4: Hand-held, Pocket (Back, Front, and Coat combined), Lower-Back, Shoulder-Bag

(New Classes — Renormalize Approach)
- MUL-SB Class 5: Hand-held, Back-pocket, Front-pocket, Coat-pocket, Lower-Back [ Unlearning Shoulder-Bag ]
- MUL-SB Class 4: Hand-held, Back-pocket (combined with Lower-Back), Front-pocket, Coat-pocket [ Unlearning Shoulder-Bag ]
- MUL-SB Class 3: Hand-held, Pocket (Lower-Back, Front-Pocket, Back-Pocket), Coat-pocket [ Unlearning Shoulder-Bag ]
- MUL-SB Class 2: Hand-held, Pocket [ Unlearning Shoulder-Bag ]

- MUL-LB Class 5: Hand-held, Back-pocket, Front-pocket, Coat-pocket, Shoulder-Bag [ Unlearning Lower-Back ]
- MUL-LB Class 4: Hand-held, Back-pocket (combined with Lower-Back), Front-pocket, Coat-pocket [ Unlearning Lower-Back ]
- MUL-LB Class 3: Hand-held, Pocket (Front-Pocket and Back-Pocket), Coat-pocket, Shoulder-Bag [ Unlearning Lower-Back ]
- MUL-LB Class 2: Hand-held, Pocket, Shoulder-Bag [ Unlearning Lower-Back ]

(New Classes — Merge Approach: LB merged into BP)
- Merge-LB Class 5: Hand-held, Back-pocket (+Lower-Back), Front-pocket, Coat-pocket, Shoulder-Bag [ Merging LB → BP ]
- Merge-LB Class 4: Hand-held, Trouser-pocket (BP+LB + FP), Coat-pocket, Shoulder-Bag [ Merging LB → BP, then BP+FP → TP ]
- Merge-LB Class 3: Hand-held, Pocket (BP+LB + FP + CP), Shoulder-Bag [ Merging LB → BP, then all pockets → P ]

In [7]:
%pip install mat-io numpy pandas scipy


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
import matio
import numpy as np
import pandas as pd

from model import get_model

model = get_model()
CLASS_NAMES = model.class_names
print("Class 6 labels:", CLASS_NAMES)

Loading model and feature list...
Loaded 50 selected features.
Loaded ensemble of 449 decision trees successfully.
Class 6 labels: ['LB', 'H', 'BP', 'FP', 'CP', 'SB']


In [9]:
DATA_DIR = "smartphone-placement-recognition/data"
LAB_DATA_PATH = f"{DATA_DIR}/customLab.mat"
FREELIVING_DATA_PATH = f"{DATA_DIR}/customFreeLiving.mat"

# Extra axial-correlation features present in the lab dataset only (see scripts/test_SPR.m)
LAB_CORR_COLS = [
    "CorrAccXY", "CorrAccXZ", "CorrAccYZ",
    "CorrGyrXY", "CorrGyrXZ", "CorrGyrYZ",
]


def normalize_label(value) -> str:
    if isinstance(value, (tuple, list, np.ndarray)):
        value = value[0] if len(value) else value
    return str(value).strip()


def load_dataset(path: str) -> pd.DataFrame:
    return matio.load_from_mat(path)["finalTable"]


def prepare_lab_data(df: pd.DataFrame, min_mean_gyr: float = 20) -> pd.DataFrame:
    df = df.drop(columns=[c for c in LAB_CORR_COLS if c in df.columns])
    return df[df["MeanGyr"] >= min_mean_gyr].copy()


def prepare_freeliving_data(df: pd.DataFrame, min_mean_gyr: float = 10) -> pd.DataFrame:
    return df[df["MeanGyr"] >= min_mean_gyr].copy()


def predict_class6(model, df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    features = df[model.selected_features].to_numpy(dtype=float)
    y_true = df["Position"].map(normalize_label).to_numpy()

    y_pred = []
    for row in features:
        scores = model.run_predictions(row)
        y_pred.append(max(scores, key=scores.get))

    return y_true, np.array(y_pred)

In [10]:
def evaluate_class6(y_true: np.ndarray, y_pred: np.ndarray, labels: list[str]) -> dict:
    label_to_idx = {label: i for i, label in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=int)

    for true_label, pred_label in zip(y_true, y_pred):
        cm[label_to_idx[true_label], label_to_idx[pred_label]] += 1

    accuracy = np.trace(cm) / cm.sum()
    per_class = {}
    for i, label in enumerate(labels):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        recall = tp / (tp + fn) if tp + fn else 0.0
        precision = tp / (tp + fp) if tp + fp else 0.0
        specificity = tn / (tn + fp) if tn + fp else 0.0
        per_class[label] = {
            "support": int(cm[i, :].sum()),
            "recall": recall,
            "precision": precision,
            "balanced_accuracy": (recall + specificity) / 2,
        }

    return {
        "accuracy": accuracy,
        "confusion_matrix": cm,
        "per_class": per_class,
    }


def print_class6_report(title: str, results: dict, labels: list[str]) -> None:
    print(f"\n=== {title} ===")
    print(f"Accuracy: {results['accuracy']:.4f}")

    print("\nPer-class metrics:")
    for label in labels:
        m = results["per_class"][label]
        print(
            f"  {label:>2} | support={m['support']:>4} | "
            f"recall={m['recall']:.3f} | precision={m['precision']:.3f} | "
            f"balanced_acc={m['balanced_accuracy']:.3f}"
        )

    cm = results["confusion_matrix"]
    header = " ".join(f"{label:>4}" for label in labels)
    print("\nConfusion matrix (rows=true, cols=pred):")
    print("     ", header)
    for i, label in enumerate(labels):
        row = " ".join(f"{cm[i, j]:>4}" for j in range(len(labels)))
        print(f"{label:>4} {row}")

## Test the original model against availabe dataset

### Test the original model aginst customLab.mat

In [ ]:
# In-lab external validation (matches scripts/test_SPR.m preprocessing)
lab_df = prepare_lab_data(load_dataset(LAB_DATA_PATH), min_mean_gyr=20)
lab_y_true, lab_y_pred = predict_class6(model, lab_df)
lab_results = evaluate_class6(lab_y_true, lab_y_pred, CLASS_NAMES)
print_class6_report("Class 6 — customLab.mat", lab_results, CLASS_NAMES)


=== Class 6 — customLab.mat ===
Accuracy: 0.7938

Per-class metrics:
  LB | support= 430 | recall=0.865 | precision=0.756 | balanced_acc=0.906
   H | support= 463 | recall=0.914 | precision=0.828 | balanced_acc=0.937
  BP | support= 460 | recall=0.717 | precision=0.846 | balanced_acc=0.845
  FP | support= 463 | recall=0.635 | precision=0.945 | balanced_acc=0.814
  CP | support= 426 | recall=0.728 | precision=0.610 | balanced_acc=0.820
  SB | support= 459 | recall=0.904 | precision=0.849 | balanced_acc=0.936

Confusion matrix (rows=true, cols=pred):
        LB    H   BP   FP   CP   SB
  LB  372   12   23    0   23    0
   H    3  423    1    0   20   16
  BP   64    6  330    2   58    0
  FP   46   13   32  294   77    1
  CP    7   34    3   15  310   57
  SB    0   23    1    0   20  415


### Test the original model aginst customFreeLiving.mat

In [12]:
# Free-living validation (same motion threshold used during training)
fl_df = prepare_freeliving_data(load_dataset(FREELIVING_DATA_PATH), min_mean_gyr=10)
fl_y_true, fl_y_pred = predict_class6(model, fl_df)
fl_results = evaluate_class6(fl_y_true, fl_y_pred, CLASS_NAMES)
print_class6_report("Class 6 — customFreeLiving.mat", fl_results, CLASS_NAMES)


=== Class 6 — customFreeLiving.mat ===
Accuracy: 0.9130

Per-class metrics:
  LB | support=2156 | recall=0.988 | precision=0.872 | balanced_acc=0.979
   H | support=2089 | recall=0.952 | precision=0.985 | balanced_acc=0.974
  BP | support=2144 | recall=0.901 | precision=0.901 | balanced_acc=0.940
  FP | support=2144 | recall=0.724 | precision=0.974 | balanced_acc=0.860
  CP | support=2034 | recall=0.939 | precision=0.825 | balanced_acc=0.950
  SB | support=2038 | recall=0.980 | precision=0.954 | balanced_acc=0.985

Confusion matrix (rows=true, cols=pred):
        LB    H   BP   FP   CP   SB
  LB 2131    3   13    0    9    0
   H    0 1988    5    0   53   43
  BP  164    1 1931   18   30    0
  FP  139    9  164 1552  280    0
  CP    9   11   29   22 1909   54
  SB    1    6    0    2   32 1997


### Test the original model aginst customLab.mat + customFreeLiving.mat

In [13]:
# Combined dataset validation (combining both lab and free-living datasets)
combined_df = pd.concat([lab_df, fl_df], ignore_index=True)
combined_y_true, combined_y_pred = predict_class6(model, combined_df)
combined_results = evaluate_class6(combined_y_true, combined_y_pred, CLASS_NAMES)
print_class6_report("Class 6 — Combined Dataset", combined_results, CLASS_NAMES)


=== Class 6 — Combined Dataset ===
Accuracy: 0.8919

Per-class metrics:
  LB | support=2586 | recall=0.968 | precision=0.853 | balanced_acc=0.967
   H | support=2552 | recall=0.945 | precision=0.953 | balanced_acc=0.968
  BP | support=2604 | recall=0.868 | precision=0.893 | balanced_acc=0.923
  FP | support=2607 | recall=0.708 | precision=0.969 | balanced_acc=0.852
  CP | support=2460 | recall=0.902 | precision=0.787 | balanced_acc=0.928
  SB | support=2497 | recall=0.966 | precision=0.934 | balanced_acc=0.976

Confusion matrix (rows=true, cols=pred):
        LB    H   BP   FP   CP   SB
  LB 2503   15   36    0   32    0
   H    3 2411    6    0   73   59
  BP  228    7 2261   20   88    0
  FP  185   22  196 1846  357    1
  CP   16   45   32   37 2219  111
  SB    1   29    1    2   52 2412


### Results

The accuracy of the model against the provided sample dataset is representative of the claims made in the paper.

summary:

- overall: 0.89

- hand-held: 0.97 (balanced precision and recall)
- back-pocket: 0.92 (balanced precision)
- front-pocket: 0.85 (higher precision)

## Machine Unlearning

"True" unlearning is not possible because of the architecture of the model. The model has learners "dependent" on each other, so trying to remove for one class will degrade the model.
So, we perform pseudo Machine UnLearning. (yes, i made that phrase up. But it is something that is done in the literature often)

The two types of Machine Unlerning (pseudo) that we focus on are:
- **Renormalize**: Remove the class from the final model output, and renormalize the remaining classes' probabilities
- **Merging**: Merging the classes together (something that the orignal paper already does with 5 class and 4 class. This should work out for LowerBack "unlearning". This is done under "Merging Approach" lower in the notebook)

In [14]:
def predict_with_unlearning(model, df, unlearn_labels, label_map):
    """
    Pseudo-MUL prediction: filter data, zero-out unlearned scores, renormalize,
    then merge classes via label_map.
    """
    # Filter out unlearned classes from data
    mask = ~df["Position"].map(normalize_label).isin(unlearn_labels)
    filtered_df = df[mask].copy()

    features = filtered_df[model.selected_features].to_numpy(dtype=float)
    y_true_raw = filtered_df["Position"].map(normalize_label).to_numpy()

    # Map true labels
    y_true = np.array([label_map[l] for l in y_true_raw])

    y_pred = []
    for row in features:
        scores = model.run_predictions(row)
        # Zero out unlearned classes and renormalize
        for ul in unlearn_labels:
            scores[ul] = 0.0
        total = sum(scores.values())
        if total > 0:
            scores = {k: v / total for k, v in scores.items()}
        # Aggregate scores by mapped label
        mapped_scores = {}
        for label, score in scores.items():
            if label in unlearn_labels:
                continue
            new_label = label_map[label]
            mapped_scores[new_label] = mapped_scores.get(new_label, 0.0) + score
        y_pred.append(max(mapped_scores, key=mapped_scores.get))

    return y_true, np.array(y_pred)

print("MUL helper functions loaded.")

MUL helper functions loaded.


### MUL-SB Tests (Unlearning Shoulder-Bag)

Remove SB from data, zero-out SB score in predictions, renormalize, then progressively merge pocket classes.

In [15]:
# MUL-SB Class 5: Unlearning Shoulder-Bag
# Classes: Hand-held, Back-pocket, Front-pocket, Coat-pocket, Lower-Back
mul_sb5_labels = ["LB", "H", "BP", "FP", "CP"]
mul_sb5_map = {"LB": "LB", "H": "H", "BP": "BP", "FP": "FP", "CP": "CP"}

mul_sb5_true, mul_sb5_pred = predict_with_unlearning(
    model, combined_df, ["SB"], mul_sb5_map
)
mul_sb5_results = evaluate_class6(mul_sb5_true, mul_sb5_pred, mul_sb5_labels)
print_class6_report("MUL-SB Class 5", mul_sb5_results, mul_sb5_labels)


=== MUL-SB Class 5 ===
Accuracy: 0.8883

Per-class metrics:
  LB | support=2586 | recall=0.968 | precision=0.853 | balanced_acc=0.963
   H | support=2552 | recall=0.958 | precision=0.962 | balanced_acc=0.975
  BP | support=2604 | recall=0.868 | precision=0.893 | balanced_acc=0.921
  FP | support=2607 | recall=0.708 | precision=0.970 | balanced_acc=0.851
  CP | support=2460 | recall=0.944 | precision=0.802 | balanced_acc=0.944

Confusion matrix (rows=true, cols=pred):
        LB    H   BP   FP   CP
  LB 2503   15   36    0   32
   H    3 2446    7    0   96
  BP  228    7 2261   20   88
  FP  185   22  196 1846  358
  CP   16   52   32   38 2322


In [16]:
# MUL-SB Class 4: Unlearning Shoulder-Bag, merging Back-pocket + Lower-Back
# Classes: Hand-held, Back-pocket (+LB), Front-pocket, Coat-pocket
mul_sb4_labels = ["H", "BP", "FP", "CP"]
mul_sb4_map = {"LB": "BP", "H": "H", "BP": "BP", "FP": "FP", "CP": "CP"}

mul_sb4_true, mul_sb4_pred = predict_with_unlearning(
    model, combined_df, ["SB"], mul_sb4_map
)
mul_sb4_results = evaluate_class6(mul_sb4_true, mul_sb4_pred, mul_sb4_labels)
print_class6_report("MUL-SB Class 4", mul_sb4_results, mul_sb4_labels)


=== MUL-SB Class 4 ===
Accuracy: 0.9126

Per-class metrics:
   H | support=2552 | recall=0.957 | precision=0.969 | balanced_acc=0.975
  BP | support=5190 | recall=0.986 | precision=0.890 | balanced_acc=0.951
  FP | support=2607 | recall=0.705 | precision=0.972 | balanced_acc=0.850
  CP | support=2460 | recall=0.933 | precision=0.867 | balanced_acc=0.950

Confusion matrix (rows=true, cols=pred):
         H   BP   FP   CP
   H 2442   20    0   90
  BP   12 5115   15   48
  FP   15  540 1837  215
  CP   51   75   38 2296


In [17]:
# MUL-SB Class 3: Unlearning Shoulder-Bag, Pocket = LB + FP + BP
# Classes: Hand-held, Pocket, Coat-pocket
mul_sb3_labels = ["H", "P", "CP"]
mul_sb3_map = {"LB": "P", "H": "H", "BP": "P", "FP": "P", "CP": "CP"}

mul_sb3_true, mul_sb3_pred = predict_with_unlearning(
    model, combined_df, ["SB"], mul_sb3_map
)
mul_sb3_results = evaluate_class6(mul_sb3_true, mul_sb3_pred, mul_sb3_labels)
print_class6_report("MUL-SB Class 3", mul_sb3_results, mul_sb3_labels)


=== MUL-SB Class 3 ===
Accuracy: 0.9563

Per-class metrics:
   H | support=2552 | recall=0.956 | precision=0.969 | balanced_acc=0.974
   P | support=7797 | recall=0.970 | precision=0.975 | balanced_acc=0.966
  CP | support=2460 | recall=0.913 | precision=0.885 | balanced_acc=0.942

Confusion matrix (rows=true, cols=pred):
         H    P   CP
   H 2439   28   85
   P   26 7565  206
  CP   51  164 2245


In [18]:
# MUL-SB Class 2: Unlearning Shoulder-Bag, everything else = Pocket
# Classes: Hand-held, Pocket
mul_sb2_labels = ["H", "P"]
mul_sb2_map = {"LB": "P", "H": "H", "BP": "P", "FP": "P", "CP": "P"}

mul_sb2_true, mul_sb2_pred = predict_with_unlearning(
    model, combined_df, ["SB"], mul_sb2_map
)
mul_sb2_results = evaluate_class6(mul_sb2_true, mul_sb2_pred, mul_sb2_labels)
print_class6_report("MUL-SB Class 2", mul_sb2_results, mul_sb2_labels)


=== MUL-SB Class 2 ===
Accuracy: 0.9852

Per-class metrics:
   H | support=2552 | recall=0.929 | precision=0.996 | balanced_acc=0.964
   P | support=10257 | recall=0.999 | precision=0.983 | balanced_acc=0.964

Confusion matrix (rows=true, cols=pred):
         H    P
   H 2371  181
   P    9 10248


### MUL-LB Tests (Unlearning Lower-Back)

Remove LB from data, zero-out LB score in predictions, renormalize, then progressively merge pocket classes.

In [19]:
# MUL-LB Class 5: Unlearning Lower-Back
# Classes: Hand-held, Back-pocket, Front-pocket, Coat-pocket, Shoulder-Bag
mul_lb5_labels = ["H", "BP", "FP", "CP", "SB"]
mul_lb5_map = {"H": "H", "BP": "BP", "FP": "FP", "CP": "CP", "SB": "SB"}

mul_lb5_true, mul_lb5_pred = predict_with_unlearning(
    model, combined_df, ["LB"], mul_lb5_map
)
mul_lb5_results = evaluate_class6(mul_lb5_true, mul_lb5_pred, mul_lb5_labels)
print_class6_report("MUL-LB Class 5", mul_lb5_results, mul_lb5_labels)


=== MUL-LB Class 5 ===
Accuracy: 0.8888

Per-class metrics:
   H | support=2552 | recall=0.945 | precision=0.951 | balanced_acc=0.966
  BP | support=2604 | recall=0.923 | precision=0.891 | balanced_acc=0.947
  FP | support=2607 | recall=0.708 | precision=0.969 | balanced_acc=0.851
  CP | support=2460 | recall=0.907 | precision=0.744 | balanced_acc=0.916
  SB | support=2497 | recall=0.966 | precision=0.934 | balanced_acc=0.975

Confusion matrix (rows=true, cols=pred):
         H   BP   FP   CP   SB
   H 2412    7    0   74   59
  BP    9 2403   20  172    0
  FP   40  251 1846  469    1
  CP   45   35   37 2232  111
  SB   30    1    2   52 2412


In [20]:
# MUL-LB Class 4: Unlearning Lower-Back (+ Shoulder-Bag)
# Classes: Hand-held, Back-pocket, Front-pocket, Coat-pocket
mul_lb4_labels = ["H", "BP", "FP", "CP"]
mul_lb4_map = {"H": "H", "BP": "BP", "FP": "FP", "CP": "CP"}

mul_lb4_true, mul_lb4_pred = predict_with_unlearning(
    model, combined_df, ["LB", "SB"], mul_lb4_map
)
mul_lb4_results = evaluate_class6(mul_lb4_true, mul_lb4_pred, mul_lb4_labels)
print_class6_report("MUL-LB Class 4", mul_lb4_results, mul_lb4_labels)


=== MUL-LB Class 4 ===
Accuracy: 0.8834

Per-class metrics:
   H | support=2552 | recall=0.959 | precision=0.960 | balanced_acc=0.973
  BP | support=2604 | recall=0.923 | precision=0.891 | balanced_acc=0.942
  FP | support=2607 | recall=0.708 | precision=0.970 | balanced_acc=0.850
  CP | support=2460 | recall=0.949 | precision=0.760 | balanced_acc=0.927

Confusion matrix (rows=true, cols=pred):
         H   BP   FP   CP
   H 2447    8    0   97
  BP    9 2403   20  172
  FP   40  251 1846  470
  CP   52   35   38 2335


In [21]:
# MUL-LB Class 3: Unlearning Lower-Back, Pocket = FP + BP
# Classes: Hand-held, Pocket, Coat-pocket, Shoulder-Bag
mul_lb3_labels = ["H", "P", "CP", "SB"]
mul_lb3_map = {"H": "H", "BP": "P", "FP": "P", "CP": "CP", "SB": "SB"}

mul_lb3_true, mul_lb3_pred = predict_with_unlearning(
    model, combined_df, ["LB"], mul_lb3_map
)
mul_lb3_results = evaluate_class6(mul_lb3_true, mul_lb3_pred, mul_lb3_labels)
print_class6_report("MUL-LB Class 3", mul_lb3_results, mul_lb3_labels)


=== MUL-LB Class 3 ===
Accuracy: 0.9137

Per-class metrics:
   H | support=2552 | recall=0.944 | precision=0.952 | balanced_acc=0.966
   P | support=5211 | recall=0.887 | precision=0.969 | balanced_acc=0.933
  CP | support=2460 | recall=0.887 | precision=0.768 | balanced_acc=0.912
  SB | support=2497 | recall=0.964 | precision=0.935 | balanced_acc=0.974

Confusion matrix (rows=true, cols=pred):
         H    P   CP   SB
   H 2410   15   69   58
   P   48 4622  540    1
  CP   45  123 2183  109
  SB   29   12   49 2407


In [22]:
# MUL-LB Class 2: Unlearning Lower-Back, Pocket = FP + BP + CP
# Classes: Hand-held, Pocket, Shoulder-Bag
mul_lb2_labels = ["H", "P", "SB"]
mul_lb2_map = {"H": "H", "BP": "P", "FP": "P", "CP": "P", "SB": "SB"}

mul_lb2_true, mul_lb2_pred = predict_with_unlearning(
    model, combined_df, ["LB"], mul_lb2_map
)
mul_lb2_results = evaluate_class6(mul_lb2_true, mul_lb2_pred, mul_lb2_labels)
print_class6_report("MUL-LB Class 2", mul_lb2_results, mul_lb2_labels)


=== MUL-LB Class 2 ===
Accuracy: 0.9660

Per-class metrics:
   H | support=2552 | recall=0.928 | precision=0.974 | balanced_acc=0.961
   P | support=7671 | recall=0.983 | precision=0.971 | balanced_acc=0.969
  SB | support=2497 | recall=0.954 | precision=0.943 | balanced_acc=0.970

Confusion matrix (rows=true, cols=pred):
         H    P   SB
   H 2368  133   51
   P   41 7538   92
  SB   23   92 2382


### Merge-LB Tests (Merging Lower-Back → Back-pocket)

Instead of zeroing out LB scores (renormalize), we **add** LB probability to BP.
True labels for LB samples are remapped to BP. No data is filtered — all samples are used.

This is the **merge** approach to pseudo-unlearning, vs the **renormalize** approach used in MUL-LB above.

In [23]:
def predict_with_merging(model, df, label_map):
    """
    Pseudo-MUL via merging: remap true labels and aggregate model scores
    by the merged label. No data is filtered; no scores are zeroed out.
    """
    features = df[model.selected_features].to_numpy(dtype=float)
    y_true_raw = df["Position"].map(normalize_label).to_numpy()

    # Map true labels
    y_true = np.array([label_map[l] for l in y_true_raw])

    y_pred = []
    for row in features:
        scores = model.run_predictions(row)
        # Aggregate scores by mapped label (LB score added to BP, etc.)
        mapped_scores = {}
        for label, score in scores.items():
            new_label = label_map[label]
            mapped_scores[new_label] = mapped_scores.get(new_label, 0.0) + score
        y_pred.append(max(mapped_scores, key=mapped_scores.get))

    return y_true, np.array(y_pred)

print("Merge helper function loaded.")

Merge helper function loaded.


In [24]:
# Merge-LB Class 5: Merge LB into BP
# Classes: Hand-held, Back-pocket (+LB), Front-pocket, Coat-pocket, Shoulder-Bag
merge_lb5_labels = ["H", "BP", "FP", "CP", "SB"]
merge_lb5_map = {"LB": "BP", "H": "H", "BP": "BP", "FP": "FP", "CP": "CP", "SB": "SB"}

merge_lb5_true, merge_lb5_pred = predict_with_merging(
    model, combined_df, merge_lb5_map
)
merge_lb5_results = evaluate_class6(merge_lb5_true, merge_lb5_pred, merge_lb5_labels)
print_class6_report("Merge-LB Class 5", merge_lb5_results, merge_lb5_labels)


=== Merge-LB Class 5 ===
Accuracy: 0.9123

Per-class metrics:
   H | support=2552 | recall=0.943 | precision=0.960 | balanced_acc=0.968
  BP | support=5190 | recall=0.986 | precision=0.889 | balanced_acc=0.961
  FP | support=2607 | recall=0.705 | precision=0.971 | balanced_acc=0.850
  CP | support=2460 | recall=0.892 | precision=0.852 | balanced_acc=0.931
  SB | support=2497 | recall=0.966 | precision=0.934 | balanced_acc=0.976

Confusion matrix (rows=true, cols=pred):
         H   BP   FP   CP   SB
   H 2407   20    0   67   58
  BP   12 5115   15   48    0
  FP   15  540 1837  214    1
  CP   44   75   37 2194  110
  SB   29    3    2   52 2411


In [25]:
# Merge-LB Class 4: Merge LB→BP, then merge FP+BP→TP (Trouser-pocket)
# Classes: Hand-held, Trouser-pocket (BP+LB+FP), Coat-pocket, Shoulder-Bag
merge_lb4_labels = ["H", "TP", "CP", "SB"]
merge_lb4_map = {"LB": "TP", "H": "H", "BP": "TP", "FP": "TP", "CP": "CP", "SB": "SB"}

merge_lb4_true, merge_lb4_pred = predict_with_merging(
    model, combined_df, merge_lb4_map
)
merge_lb4_results = evaluate_class6(merge_lb4_true, merge_lb4_pred, merge_lb4_labels)
print_class6_report("Merge-LB Class 4", merge_lb4_results, merge_lb4_labels)


=== Merge-LB Class 4 ===
Accuracy: 0.9486

Per-class metrics:
   H | support=2552 | recall=0.942 | precision=0.961 | balanced_acc=0.967
  TP | support=7797 | recall=0.970 | precision=0.974 | balanced_acc=0.972
  CP | support=2460 | recall=0.872 | precision=0.871 | balanced_acc=0.923
  SB | support=2497 | recall=0.964 | precision=0.935 | balanced_acc=0.975

Confusion matrix (rows=true, cols=pred):
         H   TP   CP   SB
   H 2404   27   63   58
  TP   26 7565  205    1
  CP   44  164 2144  108
  SB   28   13   49 2407


In [26]:
# Merge-LB Class 3: Merge LB→BP, then merge all pockets→P
# Classes: Hand-held, Pocket (BP+LB+FP+CP), Shoulder-Bag
merge_lb3_labels = ["H", "P", "SB"]
merge_lb3_map = {"LB": "P", "H": "H", "BP": "P", "FP": "P", "CP": "P", "SB": "SB"}

merge_lb3_true, merge_lb3_pred = predict_with_merging(
    model, combined_df, merge_lb3_map
)
merge_lb3_results = evaluate_class6(merge_lb3_true, merge_lb3_pred, merge_lb3_labels)
print_class6_report("Merge-LB Class 3", merge_lb3_results, merge_lb3_labels)


=== Merge-LB Class 3 ===
Accuracy: 0.9721

Per-class metrics:
   H | support=2552 | recall=0.918 | precision=0.989 | balanced_acc=0.958
   P | support=10257 | recall=0.990 | precision=0.975 | balanced_acc=0.970
  SB | support=2497 | recall=0.952 | precision=0.943 | balanced_acc=0.971

Confusion matrix (rows=true, cols=pred):
         H    P   SB
   H 2344  157   51
   P    8 10157   92
  SB   18  101 2378


### Merge-LB Tests (Unlearning Lower-Back by Merging)

Keep LB data and probabilities, but map/merge LB into Back-pocket (BP) or broader Pocket classes.

In [27]:
# Merge-LB Class 5: Unlearning Lower-Back by merging into Back-pocket
# Classes: Hand-held, Back-pocket (+LB), Front-pocket, Coat-pocket, Shoulder-Bag
merge_lb5_labels = ["H", "BP", "FP", "CP", "SB"]
merge_lb5_map = {"LB": "BP", "H": "H", "BP": "BP", "FP": "FP", "CP": "CP", "SB": "SB"}

merge_lb5_true, merge_lb5_pred = predict_with_unlearning(
    model, combined_df, [], merge_lb5_map
)
merge_lb5_results = evaluate_class6(merge_lb5_true, merge_lb5_pred, merge_lb5_labels)
print_class6_report("Merge-LB Class 5", merge_lb5_results, merge_lb5_labels)


=== Merge-LB Class 5 ===
Accuracy: 0.9123

Per-class metrics:
   H | support=2552 | recall=0.943 | precision=0.960 | balanced_acc=0.968
  BP | support=5190 | recall=0.986 | precision=0.889 | balanced_acc=0.961
  FP | support=2607 | recall=0.705 | precision=0.971 | balanced_acc=0.850
  CP | support=2460 | recall=0.892 | precision=0.852 | balanced_acc=0.931
  SB | support=2497 | recall=0.966 | precision=0.934 | balanced_acc=0.976

Confusion matrix (rows=true, cols=pred):
         H   BP   FP   CP   SB
   H 2407   20    0   67   58
  BP   12 5115   15   48    0
  FP   15  540 1837  214    1
  CP   44   75   37 2194  110
  SB   29    3    2   52 2411


In [28]:
# Merge-LB Class 4: Merging LB → BP, then BP+FP → TP (Trouser-pocket)
# Classes: Hand-held, Trouser-pocket, Coat-pocket, Shoulder-Bag
merge_lb4_labels = ["H", "TP", "CP", "SB"]
merge_lb4_map = {"LB": "TP", "H": "H", "BP": "TP", "FP": "TP", "CP": "CP", "SB": "SB"}

merge_lb4_true, merge_lb4_pred = predict_with_unlearning(
    model, combined_df, [], merge_lb4_map
)
merge_lb4_results = evaluate_class6(merge_lb4_true, merge_lb4_pred, merge_lb4_labels)
print_class6_report("Merge-LB Class 4", merge_lb4_results, merge_lb4_labels)


=== Merge-LB Class 4 ===
Accuracy: 0.9486

Per-class metrics:
   H | support=2552 | recall=0.942 | precision=0.961 | balanced_acc=0.967
  TP | support=7797 | recall=0.970 | precision=0.974 | balanced_acc=0.972
  CP | support=2460 | recall=0.872 | precision=0.871 | balanced_acc=0.923
  SB | support=2497 | recall=0.964 | precision=0.935 | balanced_acc=0.975

Confusion matrix (rows=true, cols=pred):
         H   TP   CP   SB
   H 2404   27   63   58
  TP   26 7565  205    1
  CP   44  164 2144  108
  SB   28   13   49 2407


In [29]:
# Merge-LB Class 3: Merging LB → BP, then all pockets → P (Pocket)
# Classes: Hand-held, Pocket, Shoulder-Bag
merge_lb3_labels = ["H", "P", "SB"]
merge_lb3_map = {"LB": "P", "H": "H", "BP": "P", "FP": "P", "CP": "P", "SB": "SB"}

merge_lb3_true, merge_lb3_pred = predict_with_unlearning(
    model, combined_df, [], merge_lb3_map
)
merge_lb3_results = evaluate_class6(merge_lb3_true, merge_lb3_pred, merge_lb3_labels)
print_class6_report("Merge-LB Class 3", merge_lb3_results, merge_lb3_labels)


=== Merge-LB Class 3 ===
Accuracy: 0.9721

Per-class metrics:
   H | support=2552 | recall=0.918 | precision=0.989 | balanced_acc=0.958
   P | support=10257 | recall=0.990 | precision=0.975 | balanced_acc=0.970
  SB | support=2497 | recall=0.952 | precision=0.943 | balanced_acc=0.971

Confusion matrix (rows=true, cols=pred):
         H    P   SB
   H 2344  157   51
   P    8 10157   92
  SB   18  101 2378


## Overall Findings & Summary

### 1. Baseline Performance (Combined Dataset, 6 Classes)
- **Overall Accuracy**: 89.19%
- **Strengths**: The model is highly accurate at identifying Lower-Back (Recall: 96.8%) and Shoulder-Bag (Recall: 96.6%, Precision: 93.4%).
- **Weaknesses**: Front-Pocket (FP) struggles the most, with a significantly lower recall of 70.8%, often getting confused with Coat-Pocket (CP) and Back-Pocket (BP).

---

### 2. Pseudo-Unlearning Shoulder-Bag (MUL-SB via Renormalization)
By dropping Shoulder-Bag and renormalizing the probabilities of the remaining classes:
- **Class 5** (LB, H, BP, FP, CP): Accuracy slightly drops to **88.83%**.
- **Class 4** (Merging LB → BP): Accuracy improves to **91.26%**. BP recall jumps to 98.6%.
- **Class 3** (Merging all Pockets except CP): Accuracy hits **95.63%**, showing that grouping pocket classes resolves the ambiguity between BP, FP, and LB.
- **Class 2** (Hand-held vs Pocket): Accuracy peaks at **98.52%**. The generalized "Pocket" class has a near-perfect recall of 99.9%.

---

### 3. Pseudo-Unlearning Lower-Back (MUL-LB via Renormalization)
By dropping Lower-Back and renormalizing probabilities:
- **Class 5** (H, BP, FP, CP, SB): Accuracy is **88.88%**.
- **Class 4** (Dropping SB as well): Accuracy is **88.34%**.
- **Class 3** (Merging FP → P, BP → P): Accuracy improves to **91.37%**.
- **Class 2** (Hand-held vs Pocket vs SB): Accuracy reaches **96.60%**.

---

### 4. Pseudo-Unlearning Lower-Back (Merge-LB via Merging)
Instead of dropping Lower-Back, retaining the data and directly mapping its prediction to Back-Pocket (and broader pocket categories):
- **Class 5** (Merging LB → BP): Accuracy immediately jumps to **91.23%**. This approach is clearly superior to just dropping the LB class (which yielded 88.88%).
- **Class 4** (LB → BP, then BP+FP → Trouser-Pocket): Accuracy reaches **94.86%**.
- **Class 3** (All Pockets merged into 'Pocket'): Accuracy is **97.21%**.

---

### Conclusion
Merging lower-back to back pocket = good idea
Merging all pockets and lower-back = good idea
Shoulder-bag unlearning = not needed